In [6]:
%pip install -U gradio requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import requests
import gradio as gr

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = (
    os.getenv("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_KEY")
)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL = "google/gemini-2.5-flash"

SYSTEM_PROMPT = """
You are JEKACODE, a world-class senior software engineer,
programming instructor, and code reviewer.

Explain code clearly to beginners while remaining technically accurate.

For every code explanation, provide:

1. A simple summary of what the code does.
2. A section-by-section explanation.
3. Important programming concepts.
4. Inputs and outputs.
5. Bugs, errors, or possible improvements.
6. Improved code when useful.
7. A simple usage or testing example.
8. Time and space complexity when relevant.

Rules:
- Use clear beginner-friendly language.
- Preserve the original programming language.
- Do not invent code behavior.
- If the code is incomplete, explain what is missing.
- If there is an error, identify it and explain the fix.
- Never expose API keys, passwords, tokens, or secrets.
- Use Markdown headings and code blocks.
"""


def explain_code(code, language, explanation_level, user_question):

    if not OPENROUTER_API_KEY:
        return "OpenRouter API key not found. Check your .env file."

    if not code or not code.strip():
        return "Please paste some code first."

    code = code.strip()[:12000]

    if user_question:
        question = user_question.strip()
    else:
        question = "No additional question."

    prompt = (
        "Explain the following code clearly and accurately.\n\n"
        "Programming language: " + str(language) + "\n"
        "Explanation level: " + str(explanation_level) + "\n"
        "Additional question: " + question + "\n\n"
        "Code:\n"
        "```" + str(language) + "\n"
        + code +
        "\n```\n\n"
        "Include:\n"
        "- What the code does\n"
        "- How it works\n"
        "- Important concepts\n"
        "- Inputs and outputs\n"
        "- Errors and improvements\n"
        "- Improved code when useful\n"
        "- Example usage\n"
        "- Complexity analysis when relevant\n\n"
        "Use clear Markdown headings."
    )

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.2,
        "max_tokens": 1200
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "JEKACODE Code Explainer"
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"].strip()

    except requests.exceptions.ConnectionError:
        return (
            "Network connection failed. Check your internet connection "
            "or try a phone hotspot."
        )

    except requests.exceptions.Timeout:
        return "The request timed out. Please try again."

    except requests.exceptions.HTTPError as error:
        return f"OpenRouter API error: {error}"

    except Exception as error:
        return f"Unexpected error: {error}"

In [8]:
print("API key loaded:", bool(OPENROUTER_API_KEY))
print("Function ready:", callable(explain_code))
print("Model:", MODEL)

API key loaded: True
Function ready: True
Model: google/gemini-2.5-flash


In [9]:
CUSTOM_CSS = """
#brand-title {
    text-align: center;
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 5px;
}

#brand-subtitle {
    text-align: center;
    font-size: 18px;
    font-weight: 600;
    margin-bottom: 8px;
}

#brand-description {
    text-align: center;
    opacity: 0.75;
    margin-bottom: 25px;
}

#code-input textarea {
    font-family: Consolas, "Courier New", monospace !important;
    font-size: 14px !important;
    line-height: 1.5 !important;
}

#result-output {
    min-height: 500px;
}
"""

EXAMPLE_CODE = """def greet(name):
    return "Hello, " + name

print(greet("Ayomide"))
"""


with gr.Blocks(
    title="JEKACODE | AI Code Explainer",
    theme=gr.themes.Soft(),
    css=CUSTOM_CSS
) as demo:

    gr.Markdown(
        """
        <div id="brand-title">JEKACODE</div>

        <div id="brand-subtitle">
        AI Dey Teach You Code - Anytime, Anywhere
        </div>

        <div id="brand-description">
        Understand code faster with your intelligent AI programming mentor.
        </div>
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### 🧑‍💻 Code Workspace")

            language_input = gr.Dropdown(
                label="Programming language",
                choices=[
                    "Python",
                    "JavaScript",
                    "HTML",
                    "CSS",
                    "Java",
                    "C",
                    "C++",
                    "C#",
                    "SQL",
                    "Other"
                ],
                value="Python"
            )

            level_input = gr.Dropdown(
                label="Explanation level",
                choices=[
                    "Beginner-friendly",
                    "Intermediate",
                    "Advanced",
                    "Interview preparation"
                ],
                value="Beginner-friendly"
            )

            code_input = gr.Textbox(
                label="Paste your code here",
                placeholder=(
                    "Paste your code here...\n\n"
                    "Example:\n"
                    "def greet(name):\n"
                    "    return 'Hello, ' + name"
                ),
                lines=20,
                elem_id="code-input"
            )

            question_input = gr.Textbox(
                label="Optional question",
                placeholder="Example: Why am I getting a NameError?",
                lines=3
            )

            with gr.Row():

                explain_button = gr.Button(
                    "🚀 Explain My Code",
                    variant="primary"
                )

                example_button = gr.Button("Load Example")

                clear_button = gr.Button("Clear")

        with gr.Column(scale=1):

            gr.Markdown("### 🤖 AI Explanation")

            result_output = gr.Markdown(
                """
                ### Welcome to JEKACODE 👋

                Paste your code on the left and click
                **Explain My Code** to receive your explanation.

                You will get:

                - A simple explanation
                - Important concepts
                - Inputs and outputs
                - Errors and improvements
                - Example usage
                - Complexity analysis when relevant
                """,
                elem_id="result-output"
            )

    gr.Markdown(
        """
        <div style="text-align:center; opacity:0.65; font-size:13px;">
        Built with JEKACODE Africa • Powered by AI
        </div>
        """
    )

    explain_button.click(
        fn=explain_code,
        inputs=[
            code_input,
            language_input,
            level_input,
            question_input
        ],
        outputs=result_output
    )

    example_button.click(
        fn=lambda: EXAMPLE_CODE,
        inputs=None,
        outputs=code_input
    )

    clear_button.click(
        fn=lambda: (
            "",
            "",
            "Paste your code on the left and click **Explain My Code**."
        ),
        inputs=None,
        outputs=[
            code_input,
            question_input,
            result_output
        ]
    )

C:\Users\ELEAZAR GIDEON\AppData\Local\Temp\ipykernel_15796\3475452248.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [10]:
demo.launch(inbrowser=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
